In this dataset the unit is the user, since it's a pre-aggregated assignment table, one row per user with their variant and outcome, so I deduplicated at the user level and removed users with conflicting group assignments. If this were raw e-commerce clickstream, the unit would be different: I'd sessionize first (typically a 30-minute inactivity window defining a visit), decide whether the analysis unit is the visit or the user, and make sure randomization happened at the right level. Mismatching the randomization unit and the analysis unit is a classic A/B testing error.

In [2]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv('ab_data.csv')
data.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [4]:
print("Rows:", len(data))
print("\nGroup sizes:")
print(data['group'].value_counts())
print("\nLanding page counts:")
print(data['landing_page'].value_counts())

Rows: 294478

Group sizes:
group
treatment    147276
control      147202
Name: count, dtype: int64

Landing page counts:
landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64


In [5]:
# In a clean experiment: control should ALWAYS see old_page, treatment ALWAYS new_page.
# Let's check how often that's violated.
mismatch = data[
    ((data['group'] == 'control') & (data['landing_page'] == 'new_page')) |
    ((data['group'] == 'treatment') & (data['landing_page'] == 'old_page'))
]
print("Mismatched rows (group doesn't match page shown):", len(mismatch))
print("That's", f"{len(mismatch)/len(data)*100:.2f}%", "of the data")

Mismatched rows (group doesn't match page shown): 3893
That's 1.32% of the data


In [6]:
print("Duplicate user_ids:", data['user_id'].duplicated().sum())
# Look at one to understand it
dupe_ids = data[data['user_id'].duplicated(keep=False)]['user_id'].unique()
print("Example duplicated user:")
print(data[data['user_id'] == dupe_ids[0]])

Duplicate user_ids: 3894
Example duplicated user:
        user_id                   timestamp      group landing_page  converted
22       767017  2017-01-12 22:58:14.991443    control     new_page          0
277989   767017  2017-01-08 01:31:31.456648  treatment     new_page          0


In [10]:
# Step 1: keep only rows where group and page align correctly
df_clean = data[
    ((data['group'] == 'control') & (data['landing_page'] == 'old_page')) |
    ((data['group'] == 'treatment') & (data['landing_page'] == 'new_page'))
].copy()

# Step 2: remove users assigned to MORE THAN ONE group (ambiguous, can't attribute behavior)
conflicting_users = df_clean.groupby('user_id')['group'].nunique()
conflicting_users = conflicting_users[conflicting_users > 1].index
print("Users in conflicting groups:", len(conflicting_users))
df_clean = df_clean[~df_clean['user_id'].isin(conflicting_users)].copy()

# Step 3: drop any remaining exact duplicate users (same user appearing twice in same group)
df_clean = df_clean.drop_duplicates(subset='user_id', keep='first')

print("\nRows before:", len(data), "| Rows after cleaning:", len(df_clean))
print("\nClean group sizes:")
print(df_clean['group'].value_counts())

Users in conflicting groups: 0

Rows before: 294478 | Rows after cleaning: 290584

Clean group sizes:
group
treatment    145310
control      145274
Name: count, dtype: int64


In [11]:
from scipy.stats import chisquare

counts = df_clean['group'].value_counts().values
srm_stat, srm_p = chisquare(counts)
print(f"SRM chi-square p-value: {srm_p:.4f}")
print("Split looks healthy (p > 0.01)" if srm_p > 0.01 else "Possible SRM, investigate the split")

SRM chi-square p-value: 0.9468
Split looks healthy (p > 0.01)


### Data Quality Summary

Before analyzing results, I validated the experiment data and found two issues:

- **1.32% of rows (3,893) were mismatched** — users whose assigned group didn't match the page they actually saw (e.g., a "control" user shown the new page). These rows can't be trusted, since the group label doesn't reflect the real experience, so I removed them.
- **3,894 duplicate users** appeared in the data, some assigned to both groups. I kept only each user's first record to avoid double-counting and cross-contamination.

After cleaning, the dataset has **290,584 rows** with a near-even split (145,310 treatment / 145,274 control). A **Sample Ratio Mismatch check (chi-square p = 0.95)** confirms the split is statistically balanced, so the groups are sound to compare.

**Why it matters:** a contaminated control group biases the entire result. Cleaning first means any lift I measure later reflects the page change, not a data artifact.